# 03 - Your first sketch

Sketches live on planes. Each part comes with three pre-built design
planes (XY, XZ, YZ) you can sketch on.

> **Units:** the AlibreX API works in **centimeters** internally,
> regardless of how Alibre's UI is set to display dimensions.

**Prereq:** open a fresh empty part in Alibre (File - New - Part).

## Attach to the active part

In [ ]:
from alibrex import CurrentPart
part = CurrentPart()

## Pick a plane (XY is index 0)

In [ ]:
xy_plane = part.DesignPlanes.Item(0)
xy_plane.Name

## Make a sketch on that plane

In [ ]:
sketch = part.Sketches.AddSketch(None, xy_plane, "MyRectangle")
sketch.Name

## Draw four lines - a 5 cm × 3 cm rectangle

In [ ]:
figs = sketch.Figures
figs.AddLine(0.0, 0.0, 5.0, 0.0)
figs.AddLine(5.0, 0.0, 5.0, 3.0)
figs.AddLine(5.0, 3.0, 0.0, 3.0)
figs.AddLine(0.0, 3.0, 0.0, 0.0)

## Verify

In [ ]:
sketch.Figures.Count

## What's happening under the hood

You may have noticed we never called `BeginChange()` / `EndChange()` -
but AlibreX 29 actually *requires* those brackets around figure
additions. The `alibrex` package's COM proxy auto-wraps each `Add*`
call in its own `BeginChange - Add - EndChange` mini-transaction.

So the four lines above ran as **four separate transactions** under
the hood, one per `AddLine`.

## Alternative: explicit batch (one transaction)

When you want all four lines inside a **single** transaction
(slightly more efficient, atomic regen), call `BeginChange()` /
`EndChange()` yourself. The proxy tracks depth and stays out of
the way - no double-bracket.

In [ ]:
sk2 = part.Sketches.AddSketch(None, xy_plane, "MyRectangle2")
sk2.BeginChange()
figs2 = sk2.Figures
figs2.AddLine(10.0, 0.0, 15.0, 0.0)
figs2.AddLine(15.0, 0.0, 15.0, 3.0)
figs2.AddLine(15.0, 3.0, 10.0, 3.0)
figs2.AddLine(10.0, 3.0, 10.0, 0.0)
sk2.EndChange()
sk2.Figures.Count

## Does this pattern apply to other APIs?

| API | Transaction methods | Auto-wrapped? |
|---|---|---|
| 2D Sketches (`IADSketch`) | `BeginChange` / `EndChange` |  yes |
| 3D Sketches (`IAD3DSketch`) | `BeginChange` / `BeginChangeEx` / `EndChange` |  yes |
| Parameters (`IADParameters`) | `OpenParameterTransaction` / `CloseParameterTransaction` / `CancelParameterTransaction` |  no - call explicitly |

For parameters, the proxy does **not** auto-wrap. You must open and
close the transaction yourself - see notebook **07_parameters** for
the pattern. No other AlibreX APIs use a transaction bracket.